In [1]:
import os 
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import psycopg2
import pandas as pd

load_dotenv()

HOST = os.getenv("POSTGIS_HOST")
PORT = int(os.getenv("POSTGIS_PORT"))
USER = os.getenv("POSTGIS_USER")
PASSWORD = os.getenv("POSTGIS_PASSWORD")

engine = create_engine(
    f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/incendies",
    connect_args={"options": "-csearch_path=incendies_schema,public"}
)

# Test de connexion
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        print("Connexion réussie !")
        print("Version PostgreSQL :", result.scalar())

        dbs = conn.execute(
            text("SELECT datname FROM pg_database WHERE datistemplate = false ORDER BY datname")
        ).fetchall()
        print("Bases de données :", [db[0] for db in dbs])
except Exception as e:
    print("Erreur de connexion :", e)


Connexion réussie !
Version PostgreSQL : PostgreSQL 17.5 (Debian 17.5-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit
Bases de données : ['incendies', 'postgres']


In [2]:
DATABASE_NAME="incendies"

def get_engine():
    return create_engine(
        f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE_NAME}",
        connect_args={"options": "-csearch_path=incendies,public"},
    )

engine = get_engine()

## Création de la table commune_jour

In [ ]:
ddl = [
    """
    CREATE TABLE commune_jour AS
    SELECT 
        c.id_commune,
        d.jour::date AS date_jour,
        FALSE AS has_fire
    FROM 
        commune c
    CROSS JOIN 
        generate_series(
            '2006-01-01'::date, 
            '2025-12-31'::date, 
            '1 day'::interval
        ) AS d(jour);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
ddl = [
    """
    ALTER TABLE commune_jour
    ADD CONSTRAINT pk_commune_jour PRIMARY KEY (id_commune, date_jour);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
ddl = [
    """
    ALTER TABLE commune_jour 
    ALTER COLUMN has_fire TYPE SMALLINT 
    USING (has_fire::INT);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
sql = text("""
    UPDATE commune_jour cj
    SET has_fire = 1
    FROM incendie i
    JOIN commune c ON c.code_insee = i.code_insee
    WHERE cj.id_commune = c.id_commune 
      AND cj.date_jour = i.date_premiere_alerte::date;
""")

with engine.begin() as conn:
    conn.execute(sql)